In [ ]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================
!pip install pyradiomics SimpleITK pydicom scikit-image pandas matplotlib scikit-learn


In [ ]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
preprocessed_folder = "/content/drive/MyDrive/Medical Project/PREPROCESSED DATA"
output_csv          = "/content/drive/MyDrive/Medical Project/radiomics_features.csv"
print('PREPROCESSED DATA exists:', os.path.exists(preprocessed_folder))


In [ ]:
# ============================================================
# PYRADIOMICS FEATURE EXTRACTION
# ------------------------------------------------------------
# Loops over every patient folder inside PREPROCESSED DATA/.
# Each folder must contain:
#   preprocessed_volume.nii.gz  -- normalized CT
#   tumor_mask.nii.gz           -- binary tumor mask
#
# Extracts:
#   1. First-Order Features
#   2. Shape Features
#   3. Texture Features (GLCM, GLRLM, GLSZM, GLDM, NGTDM)
#   4. Filter-Based Features (Wavelet, LoG, Gradient, ...)
# ============================================================

import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
from tqdm import tqdm
from radiomics import featureextractor
import logging

logging.getLogger('radiomics').setLevel(logging.ERROR)

preprocessed_folder = "/content/drive/MyDrive/Medical Project/PREPROCESSED DATA"
output_csv          = "/content/drive/MyDrive/Medical Project/radiomics_features.csv"

# ============================================================
# COLLECT VALID PATIENT FOLDERS
# ============================================================
cases = []
for patient_id in sorted(os.listdir(preprocessed_folder)):
    patient_dir = os.path.join(preprocessed_folder, patient_id)
    ct_path     = os.path.join(patient_dir, 'preprocessed_volume.nii.gz')
    mask_path   = os.path.join(patient_dir, 'tumor_mask.nii.gz')
    if os.path.isdir(patient_dir) and os.path.exists(ct_path) and os.path.exists(mask_path):
        cases.append((patient_id, ct_path, mask_path))

print(f'Cases ready for extraction: {len(cases)}')

# ============================================================
# VISUALIZE FIRST CASE (CT slice + mask overlay)
# ============================================================
if cases:
    sample_id, sample_ct, sample_mask = cases[0]
    image_array = sitk.GetArrayFromImage(sitk.ReadImage(sample_ct))
    mask_array  = sitk.GetArrayFromImage(sitk.ReadImage(sample_mask))
    slice_idx   = image_array.shape[0] // 2

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image_array[slice_idx], cmap='gray')
    plt.title(f'CT Slice\n{sample_id}')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(image_array[slice_idx], cmap='gray')
    plt.imshow(mask_array[slice_idx], alpha=0.5, cmap='Reds')
    plt.title('Mask Overlay')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# ============================================================
# CONFIGURE PYRADIOMICS EXTRACTOR
# ============================================================
params = {
    'binWidth': 25,
    'resampledPixelSpacing': None,
    'interpolator': sitk.sitkBSpline,
    'verbose': False,
}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)

# ------------------------------------------------------------
# FIRST-ORDER FEATURES
# Intensity distribution: Mean, Median, Entropy, Skewness, Kurtosis, Energy
# Biological meaning: tumor brightness, density, necrosis, heterogeneity
# ------------------------------------------------------------
extractor.enableFeatureClassByName('firstorder')

# ------------------------------------------------------------
# SHAPE FEATURES
# Tumor geometry: Volume, Surface Area, Sphericity, Elongation
# Biological meaning: irregular shapes -> aggressive/invasive cancer
# ------------------------------------------------------------
extractor.enableFeatureClassByName('shape')

# ------------------------------------------------------------
# TEXTURE FEATURES
# ------------------------------------------------------------
extractor.enableFeatureClassByName('glcm')    # Gray-Level Co-occurrence Matrix
extractor.enableFeatureClassByName('glrlm')   # Gray-Level Run Length Matrix
extractor.enableFeatureClassByName('glszm')   # Gray-Level Size Zone Matrix
extractor.enableFeatureClassByName('gldm')    # Gray-Level Dependence Matrix
extractor.enableFeatureClassByName('ngtdm')   # Neighboring Gray Tone Difference Matrix

# ------------------------------------------------------------
# FILTER-BASED FEATURES
# ------------------------------------------------------------
extractor.enableImageTypeByName('Wavelet')                                       # Multi-scale frequency decomposition
extractor.enableImageTypeByName('LoG', customArgs={'sigma': [1.0, 2.0, 3.0]})   # Edge + blob detection
extractor.enableImageTypeByName('Square')
extractor.enableImageTypeByName('SquareRoot')
extractor.enableImageTypeByName('Logarithm')
extractor.enableImageTypeByName('Exponential')
extractor.enableImageTypeByName('Gradient')                                      # Intensity transitions

# ============================================================
# EXTRACT FEATURES -- BATCH LOOP
# ============================================================
all_rows = []
failed   = []

for patient_id, ct_path, mask_path in tqdm(cases, desc='Extracting features'):
    try:
        image  = sitk.ReadImage(ct_path)
        mask   = sitk.ReadImage(mask_path)
        result = extractor.execute(image, mask)

        row = {'PatientID': patient_id}
        for key, value in result.items():
            if key.startswith('diagnostics'):
                continue
            row[key] = float(value) if hasattr(value, '__float__') else value
        all_rows.append(row)

    except Exception as e:
        print(f'  FAILED [{patient_id}]: {e}')
        failed.append(patient_id)

# ============================================================
# BUILD DATAFRAME + SAVE
# ============================================================
df = pd.DataFrame(all_rows)
df.to_csv(output_csv, index=False)

print(f'\nExtracted features from {len(all_rows)} patients.')
print(f'Failed: {len(failed)} patients.')
print(f'Total features per patient: {len(df.columns) - 1}')
print(f'Saved to: {output_csv}')
print()
print(df.head())

# ============================================================
# FEATURE CLASS EXPLANATION
# ============================================================
print('\n================================================')
print('FEATURE CLASSES EXTRACTED')
print('================================================')
print()
print('1. FIRST-ORDER FEATURES')
print('   Mean, Median, Entropy, Energy, Skewness, Kurtosis')
print('   -> Tumor brightness, density, necrosis, intensity heterogeneity')
print()
print('2. SHAPE FEATURES')
print('   Volume, Surface Area, Sphericity, Compactness, Elongation')
print('   -> Irregular shapes linked to aggressive/invasive cancers')
print()
print('3. TEXTURE FEATURES')
print('   GLCM  - Spatial pixel relationships')
print('   GLRLM - Continuous intensity runs')
print('   GLSZM - Homogeneous region sizes')
print('   GLDM  - Voxel dependence patterns')
print('   NGTDM - Local contrast and coarseness')
print('   -> Tumor heterogeneity and internal structural irregularity')
print()
print('4. FILTER-BASED FEATURES')
print('   Wavelet   - Multi-scale frequency decomposition')
print('   LoG       - Edge enhancement and blob detection')
print('   Gradient  - Intensity transitions')
print('   -> Hidden tumor patterns not visible in original images')


In [ ]:
# ============================================================
# FEATURE SELECTION
# ------------------------------------------------------------
# Step 1 -- Variance Threshold  (remove near-zero variance features)
# Step 2 -- LASSO               (L1 regularisation selection)
# Step 3 -- PCA                 (dimensionality reduction to 95% variance)
#
# NOTE: LASSO requires a target variable (y).
# Replace y_proxy below with your actual malignancy labels
# when available (e.g., 0 = benign, 1 = malignant).
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LassoCV
from sklearn.decomposition import PCA

features_csv  = "/content/drive/MyDrive/Medical Project/radiomics_features.csv"
output_folder = "/content/drive/MyDrive/Medical Project"

# ============================================================
# LOAD FEATURES
# ============================================================
df = pd.read_csv(features_csv)
print(f'Loaded: {df.shape[0]} patients, {df.shape[1]-1} features')

patient_ids   = df['PatientID'].values
X             = df.drop(columns=['PatientID']).values.astype(np.float64)
feature_names = np.array(df.drop(columns=['PatientID']).columns)

# Drop all-NaN columns
nan_mask      = ~np.all(np.isnan(X), axis=0)
X             = X[:, nan_mask]
feature_names = feature_names[nan_mask]
print(f'After dropping all-NaN columns: {X.shape[1]} features remain')

# Fill remaining NaNs with column mean
col_means     = np.nanmean(X, axis=0)
nan_idx       = np.where(np.isnan(X))
X[nan_idx]    = np.take(col_means, nan_idx[1])

# ============================================================
# STEP 1 -- VARIANCE THRESHOLD
# Remove features with near-zero variance (carry no information)
# ============================================================
vt            = VarianceThreshold(threshold=0.01)
X_vt          = vt.fit_transform(X)
feature_names_vt = feature_names[vt.get_support()]
print(f'\nAfter Variance Threshold : {X_vt.shape[1]} features  (removed {X.shape[1] - X_vt.shape[1]})')

# ============================================================
# STEP 2 -- STANDARDIZE
# ============================================================
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_vt)

# ============================================================
# STEP 3 -- LASSO FEATURE SELECTION
# ============================================================
print('\nRunning LassoCV...')

# --- REPLACE WITH REAL LABELS WHEN AVAILABLE ---
y_proxy = np.nanmean(X_vt, axis=1)   # proxy target until labels exist
# ------------------------------------------------

lasso            = LassoCV(cv=5, max_iter=10000, n_jobs=-1)
lasso.fit(X_scaled, y_proxy)
lasso_mask       = lasso.coef_ != 0
X_lasso          = X_scaled[:, lasso_mask]
feature_names_lasso = feature_names_vt[lasso_mask]

print(f'Alpha chosen by CV       : {lasso.alpha_:.6f}')
print(f'After LASSO              : {X_lasso.shape[1]} features  (removed {X_vt.shape[1] - X_lasso.shape[1]})')

# Plot top-30 LASSO coefficients
top_n    = min(30, len(feature_names_lasso))
coef_abs = np.abs(lasso.coef_[lasso_mask])
top_idx  = np.argsort(coef_abs)[-top_n:][::-1]
plt.figure(figsize=(12, 5))
plt.barh(feature_names_lasso[top_idx][::-1], coef_abs[top_idx][::-1])
plt.xlabel('Absolute LASSO Coefficient')
plt.title(f'Top {top_n} Features Selected by LASSO')
plt.tight_layout()
plt.show()

# ============================================================
# STEP 4 -- PCA (95% variance explained)
# ============================================================
pca   = PCA(n_components=0.95, svd_solver='full')
X_pca = pca.fit_transform(X_lasso)
print(f'\nAfter PCA (95% variance) : {X_pca.shape[1]} components')

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_) * 100, marker='o', markersize=4)
plt.axhline(95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cumulative Explained Variance (%)')
plt.title('PCA -- Explained Variance')
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# SAVE RESULTS
# ============================================================
df_lasso = pd.DataFrame(X_lasso, columns=feature_names_lasso)
df_lasso.insert(0, 'PatientID', patient_ids)
lasso_path = os.path.join(output_folder, 'features_lasso_selected.csv')
df_lasso.to_csv(lasso_path, index=False)
print(f'\nLASSO-selected features saved to : {lasso_path}')

pca_cols = [f'PC{i+1}' for i in range(X_pca.shape[1])]
df_pca   = pd.DataFrame(X_pca, columns=pca_cols)
df_pca.insert(0, 'PatientID', patient_ids)
pca_path = os.path.join(output_folder, 'features_pca_reduced.csv')
df_pca.to_csv(pca_path, index=False)
print(f'PCA-reduced features saved to    : {pca_path}')

print(f'\nSummary:')
print(f'  Original features   : {X.shape[1]}')
print(f'  After Var Threshold : {X_vt.shape[1]}')
print(f'  After LASSO         : {X_lasso.shape[1]}')
print(f'  After PCA           : {X_pca.shape[1]} components')
